# CS781 Capstone - Phase 1: Legal LM Pretraining
## Architecture 2, continued pretraining (LLaMA-style decoder: RoPE + RMSNorm + SwiGLU + GQA)

Continues the trained Arch 2 (61.6M params, 300M tokens, val loss 2.2833, CodaBench 0.8334) on text it has never seen.

- **Same model**: Arch 2's architecture, model code and tokenizer, copied unchanged; starts from Arch 2's final weights, AdamW moments and GradScaler state.
- **New data only**:
   - Arch 2's data pipeline is replayed exactly (same seed, dedup and quotas) to recover its validation split and where it stopped reading each source; training text comes only from after those points.
   - Arch 2's validation set is reused unchanged, and a replay check confirms the starting model reproduces Arch 2's val loss.
- **Schedule**: re-warm 6e-5 -> 3e-4 over 500 updates, cosine to 3e-5 over a horizon measured during warm-up and fitted to `TRAIN_BUDGET_HOURS` (9h = one session).
- **Measured, not assumed**: held-out BERTScore of the original Arch 2 before training and of the continued model after, on the same 200 prompts from documents neither model trains on.
- **Safety**: checkpoint every 1000 updates (and hourly), previous checkpoint kept as fallback, emergency save on crash, stop 1.5h before the 12h limit, `model.pt` saved before generation.

**How to run**
- Attach the tokenizer dataset and the Arch 2 results (`model.pt`, `metrics.json`, `checkpoint.pt`), GPU T4, Internet on, `HF` secret, then *Save Version -> Save & Run All*.
- For a second session (`TRAIN_BUDGET_HOURS = 18`): attach the previous version's output, set `RUN_MODE = "resume"`, Save & Run All.
- Test predictions and `submission.zip` are produced in the session where training completes, with Arch 2's exact generation settings.

## 1. HuggingFace Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HuggingFace login successful.")

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import hashlib
import glob
import shutil
import numpy as np
from collections import Counter

NOTEBOOK_START = time.time()  # session clock: the platform limit counts from here

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data (same files, same order as Arch 2)
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Model: Arch 2, unchanged (must match the trained weights exactly)
MODEL_DIM = 640
MODEL_LAYERS = 12
MODEL_HEADS = 10
MODEL_KV_HEADS = 2        # GQA: 5 query heads per KV head
MODEL_FF = 1706           # SwiGLU: floor(2/3 * 4 * 640)
MODEL_SEQ_LEN = 1024
MODEL_DROPOUT = 0.0
VOCAB_SIZE = 16384
ARCH2_PARAMS = 61_604_480

# Tokenizer (same lookup as Arch 1/2)
_tok_matches = glob.glob("/kaggle/input/**/legal_bpe_16384.json", recursive=True)
TOKENIZER_PATH = _tok_matches[0] if _tok_matches else "/kaggle/input/legal-bpe-tokenizer/legal_bpe_16384.json"
print(f"Found tokenizer at: {TOKENIZER_PATH}")

# Arch 2's original data definition, replayed exactly to recover its validation split and per-source stop points
A2_TRAIN_TOKENS = 300_000_000
A2_VAL_TOKENS = 5_000_000
A2_FINAL_VAL_LOSS = 2.283268589611913      # Arch 2 metrics.json; the replay is verified against it

# Continued pretraining
NEW_TOKENS = 600_000_000   # tokens from documents Arch 2 never read; the sampler reshuffles if the run needs more
HELDOUT_PCT = 5            # of those new documents: never trained on, source of the held-out eval prompts
EVAL_N = 200               # held-out 512-word -> 128-word prompts for local BERTScore (before vs after)
BATCH_SIZE = 4
GRAD_ACCUM = 8             # 32 x 1024 = 32,768 tokens per update, as in Arch 2
START_LR = 6.04e-5         # where Arch 2's cosine schedule ended
PEAK_LR = 3e-4             # re-warm peak (half of Arch 2's 6e-4) so the trained model is not knocked off course
MIN_LR = 3e-5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
TRAIN_BUDGET_HOURS = 9.0   # active training time: 9 = one session, 18 = two sessions
MEASURE_FROM, MEASURE_TO = 50, 300
HORIZON_SAFETY = 0.95
SESSION_LIMIT_HOURS = 12.0
SESSION_RESERVE_HOURS = 1.5          # final eval, 1000 generations and saves need this much
CKPT_EVERY_UPDATES = 1000
CKPT_EVERY_MIN = 60
SNAPSHOT_EVERY = 2000               # fp16 weight snapshots for checkpoint selection after training...
SNAPSHOT_FROM_FRAC = 0.6            # ...over the last 40% of the planned updates
VAL_EVERY_UPDATES = 1000
MAX_VAL_BATCHES = 200
EVAL_ENABLED = True
LOG_EVERY = 100

# Generation: Arch 2's exact settings (and function), so the score is comparable with 0.8334
TARGET_WORDS = 128
MAX_GEN_TOKENS = 384
TEMPERATURE = 0.4
TOP_P = 0.92
TOP_K = 50
REPETITION_PENALTY = 1.15

# Run mode: "auto" resumes if a previous continuation checkpoint is attached, otherwise starts from Arch 2.
# Set "resume" for sessions 2+ so a forgotten input fails in seconds.
RUN_MODE = "auto"

OUTPUT_DIR = "/kaggle/working/capstone_arch2_continued"
INPUT_ROOT = "/kaggle/input"
LATEST_CKPT = "arch2c_latest.pt"
PREV_CKPT = "arch2c_latest_prev.pt"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def find_input(name):
    m = sorted(glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    return m[-1] if m else None

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, default=float)
    os.replace(tmp, path)

def session_elapsed():
    return time.time() - NOTEBOOK_START

def find_resume_dir():
    """Folder holding the most advanced continuation checkpoint among this session's output and all inputs."""
    dirs = set()
    for name in (LATEST_CKPT, PREV_CKPT):
        if os.path.exists(os.path.join(OUTPUT_DIR, name)):
            dirs.add(OUTPUT_DIR)
        dirs.update(os.path.dirname(p) for p in glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    best, best_u = None, -1
    for d in sorted(dirs):
        try:
            with open(os.path.join(d, "run_log.json")) as f:
                u = int(json.load(f)["update"])
        except Exception:
            u = 0
        if u > best_u:
            best, best_u = d, u
    return best, best_u

def find_arch2_dir():
    """Folder with the trained Arch 2: a model.pt whose sibling metrics.json names Arch 2's architecture.
    (An attached Arch 1 model.pt, or this notebook's own outputs, can never match.)"""
    for p in sorted(glob.glob(f"{INPUT_ROOT}/**/model.pt", recursive=True)):
        d = os.path.dirname(p)
        try:
            with open(os.path.join(d, "metrics.json")) as f:
                if json.load(f).get("architecture") == "llama_style_rope_rmsnorm_swiglu_gqa":
                    return d
        except Exception:
            continue
    return None

RESUME_DIR, RESUME_UPDATE = find_resume_dir()
ARCH2_DIR = None
if RESUME_DIR:
    print(f"Previous continuation run found: {RESUME_DIR} (update {RESUME_UPDATE:,}) -> this session RESUMES it")
elif RUN_MODE == "resume":
    raise RuntimeError("RUN_MODE='resume' but no arch2c_latest.pt is attached: add the previous version's output as an input")
else:
    ARCH2_DIR = find_arch2_dir()
    if ARCH2_DIR is None:
        raise RuntimeError("Arch 2 weights not found: attach the Arch 2 results (model.pt + metrics.json, ideally checkpoint.pt)")
    has_opt = os.path.exists(os.path.join(ARCH2_DIR, "checkpoint.pt"))
    print(f"Starting continued pretraining from Arch 2 in {ARCH2_DIR} "
          f"(optimizer state: {'checkpoint.pt' if has_opt else 'NOT FOUND, AdamW moments start fresh'})")
print("Configuration loaded.")


## 4. BPE Tokenizer (from A1, unchanged from Arch 2)

In [ ]:
%%writefile bpe_tokenizer.py
"""
BPE Tokenizer - CS781 Assignment 1 implementation, optimized for Phase 1.

Speedups over original A1 code (same algorithm):
  1. fit() uses word frequencies + incremental pair-count updates
  2. encode() uses priority-based merging + word-level cache
  3. Checkpointing: saves progress every N merges so training survives Kaggle timeouts

Research-backed defaults:
  - vocab_size=16384: optimal for 100M-scale models (Zheng et al. 2024,
    "Scaling Laws with Vocabulary"). At 100M scale, 16K minimizes training
    cost; quality spread across 8K-256K is <2% BPB.
  - Corpus sample: 5-10M words is sufficient for merge stabilization.
"""

import json
import os
import time
from collections import defaultdict


class BPETokenizer:

    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos.extend(range(256))
        self.eow_id = len(self.itos)  # 260
        self.itos.append("</w>")
        self.merges = []
        self.vocab_size = len(self.itos)
        self._merge_rank = {}
        self._encode_cache = {}
        self._token_bytes = None
        self._ends_word = None

    # ------------------------------------------------------------------ #
    #  Training (with checkpointing)                                      #
    # ------------------------------------------------------------------ #

    def fit(self, corpus, checkpoint_dir=None, checkpoint_every=1000):
        ckpt_path = None
        if checkpoint_dir:
            os.makedirs(checkpoint_dir, exist_ok=True)
            ckpt_path = os.path.join(checkpoint_dir, f"bpe_v{self.target_vocab_size}.ckpt.json")

        resumed_merge = 0
        word_data = None

        if ckpt_path and os.path.exists(ckpt_path):
            print(f"Found checkpoint at {ckpt_path}, resuming...")
            with open(ckpt_path, "r") as f:
                ckpt = json.load(f)
            self.merges = [((a, b), c) for (a, b), c in ckpt["merges"]]
            self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
            self.itos.extend(range(256))
            self.itos.append("</w>")
            for pair, new_id in self.merges:
                self.itos.append(pair)
            self.vocab_size = len(self.itos)
            resumed_merge = len(self.merges)
            word_data = []
            for tokens, freq in ckpt["words"]:
                word_data.append([tokens, freq])
            print(f"  Resumed after {resumed_merge} merges (vocab: {self.vocab_size})")

        if word_data is None:
            print("Counting word frequencies...")
            word_freq = {}
            for word in corpus.split():
                word_freq[word] = word_freq.get(word, 0) + 1
            print(f"  {len(word_freq):,} unique words")

            word_data = []
            for word, freq in word_freq.items():
                tokens = [b + 4 for b in word.encode("utf-8")]
                tokens.append(self.eow_id)
                word_data.append([tokens, freq])

        print("Building pair index...")
        pair_counts = defaultdict(int)
        pair_words = defaultdict(set)

        for idx, (tokens, freq) in enumerate(word_data):
            seen = defaultdict(int)
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                seen[pair] += 1
            for pair, cnt in seen.items():
                pair_counts[pair] += cnt * freq
                pair_words[pair].add(idx)

        num_merges_total = self.target_vocab_size - 261
        num_merges_left = num_merges_total - resumed_merge
        print(f"Learning {num_merges_left} merges (of {num_merges_total} total)...")
        t0 = time.time()

        for merge_i in range(num_merges_left):
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            if pair_counts[best_pair] < 2:
                break

            new_id = len(self.itos)
            bp_a, bp_b = best_pair

            affected = list(pair_words.get(best_pair, set()))

            for idx in affected:
                tokens, freq = word_data[idx]

                old_pairs = defaultdict(int)
                for i in range(len(tokens) - 1):
                    old_pairs[(tokens[i], tokens[i + 1])] += 1
                for p, cnt in old_pairs.items():
                    pair_counts[p] -= cnt * freq
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_words[p].discard(idx)
                    if not pair_words[p]:
                        pair_words.pop(p, None)

                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == bp_a and tokens[i + 1] == bp_b:
                        new_tokens.append(new_id)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                word_data[idx][0] = new_tokens

                new_pairs = defaultdict(int)
                for i in range(len(new_tokens) - 1):
                    new_pairs[(new_tokens[i], new_tokens[i + 1])] += 1
                for p, cnt in new_pairs.items():
                    pair_counts[p] = pair_counts.get(p, 0) + cnt * freq
                    pair_words[p].add(idx)

            self.merges.append((best_pair, new_id))
            self.itos.append(best_pair)
            self.vocab_size = len(self.itos)

            done = resumed_merge + merge_i + 1
            if done % 1000 == 0:
                elapsed = time.time() - t0
                rate = (merge_i + 1) / elapsed
                eta = (num_merges_left - merge_i - 1) / rate if rate > 0 else 0
                print(f"  {done}/{num_merges_total} merges | "
                      f"{elapsed:.0f}s elapsed | "
                      f"~{eta / 60:.1f} min remaining")

            if ckpt_path and (merge_i + 1) % checkpoint_every == 0:
                self._save_checkpoint(ckpt_path, word_data)

        if ckpt_path:
            self._save_checkpoint(ckpt_path, word_data)

        self._rebuild_caches()
        total_time = time.time() - t0
        print(f"Done. Final vocab: {self.vocab_size} | Time: {total_time / 60:.1f} min")
        return self

    def _save_checkpoint(self, path, word_data):
        ckpt = {
            "merges": [[[a, b], c] for (a, b), c in self.merges],
            "words": [[tokens, freq] for tokens, freq in word_data],
        }
        with open(path, "w") as f:
            json.dump(ckpt, f)

    # ------------------------------------------------------------------ #
    #  Encoding                                                           #
    # ------------------------------------------------------------------ #

    def _rebuild_caches(self):
        self._merge_rank = {}
        for rank, ((a, b), new_id) in enumerate(self.merges):
            self._merge_rank[(a, b)] = (rank, new_id)
        self._encode_cache = {}

        self._token_bytes = {x + 4: bytes([x]) for x in range(256)}
        self._token_bytes[self.eow_id] = b""
        self._ends_word = {i + 4: False for i in range(256)}
        self._ends_word[self.eow_id] = True
        for (a, b), c in self.merges:
            self._token_bytes[c] = self._token_bytes[a] + self._token_bytes[b]
            self._ends_word[c] = self._ends_word[b]

    def _encode_word(self, word):
        tokens = [b + 4 for b in word.encode("utf-8")]
        tokens.append(self.eow_id)

        while len(tokens) >= 2:
            best_rank = float("inf")
            best_idx = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                r = self._merge_rank.get(pair)
                if r is not None and r[0] < best_rank:
                    best_rank = r[0]
                    best_idx = i

            if best_idx == -1:
                break

            _, new_id = self._merge_rank[(tokens[best_idx], tokens[best_idx + 1])]
            tokens = tokens[:best_idx] + [new_id] + tokens[best_idx + 2:]

        return tokens

    def encode(self, text):
        if not self._merge_rank and self.merges:
            self._rebuild_caches()

        result = []
        for word in text.split():
            cached = self._encode_cache.get(word)
            if cached is not None:
                result.extend(cached)
            else:
                tokens = self._encode_word(word)
                self._encode_cache[word] = tokens
                result.extend(tokens)
        return result

    # ------------------------------------------------------------------ #
    #  Decoding                                                           #
    # ------------------------------------------------------------------ #

    def decode(self, tokens):
        if self._token_bytes is None:
            self._rebuild_caches()

        parts = []
        for i, token in enumerate(tokens):
            b = self._token_bytes.get(token)
            if b is not None:
                parts.append(b)
                if self._ends_word.get(token, False) and i < len(tokens) - 1:
                    parts.append(b" ")

        return b"".join(parts).decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ #
    #  Save / Load                                                        #
    # ------------------------------------------------------------------ #

    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "target_vocab_size": self.target_vocab_size,
            "merges": [[[a, b], c] for (a, b), c in self.merges],
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"Tokenizer saved to {path} ({self.vocab_size} tokens)")

    @classmethod
    def load(cls, path):
        with open(path, "r") as f:
            data = json.load(f)
        tok = cls(data["target_vocab_size"])
        tok.merges = [((a, b), c) for (a, b), c in data["merges"]]
        for pair, new_id in tok.merges:
            tok.itos.append(pair)
        tok.vocab_size = len(tok.itos)
        tok._rebuild_caches()
        return tok

    @property
    def pad_token_id(self):
        return 0

    @property
    def unk_token_id(self):
        return 1

    @property
    def bos_token_id(self):
        return 2

    @property
    def eos_token_id(self):
        return 3

    def __len__(self):
        return self.vocab_size

## 5. Load Trained Tokenizer

In [ ]:
from bpe_tokenizer import BPETokenizer

tokenizer = BPETokenizer.load(TOKENIZER_PATH)
print(f"Loaded tokenizer: vocab_size={tokenizer.vocab_size}")

# sanity check
test_text = "The Supreme Court of India hereby declares"
enc = tokenizer.encode(test_text)
dec = tokenizer.decode(enc)
print(f"Encode: {test_text!r} -> {enc[:15]}...")
print(f"Decode: {dec!r}")
print(f"Tokens per word: {len(enc)/len(test_text.split()):.2f}")

## 6. Data: Replay Arch 2, then Collect New Text

1. **Replay**: stream each source exactly as Arch 2 did (same `random` seed, per-source dedup, <50-word filter, 5% validation draw, 75M train / 1.25M val quotas) and record the document where Arch 2 stopped.
   - Arch 2's validation tokens are rebuilt identically.
2. **New region**: documents after the stop point (skipping any document Arch 2 read in this or an earlier source). 5% are held out entirely and supply the eval prompts; the rest give up to `NEW_TOKENS` new training tokens.
3. Everything is cached to the output folder; later sessions reload it.

In [ ]:
try:
    import ijson
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'ijson', '-q'], check=True)
    import ijson

import array
import gc
from huggingface_hub import hf_hub_download

BLOCK = MODEL_SEQ_LEN + 1      # 1025 tokens -> (input 1024, target 1024), exactly as Arch 2's TokenBlockDataset
CACHE = {
    "train": "arch2c_train_blocks.npy",
    "val": "arch2c_val_blocks.npy",
    "eval": "arch2c_eval_set.json",
    "meta": "arch2c_data_meta.json",
}

def find_cached(name):
    if RESUME_DIR and os.path.exists(os.path.join(RESUME_DIR, name)):
        return os.path.join(RESUME_DIR, name)
    local = os.path.join(OUTPUT_DIR, name)
    return local if os.path.exists(local) else find_input(name)

cached = {k: find_cached(v) for k, v in CACHE.items()}
if RESUME_DIR and not all(cached.values()):
    raise RuntimeError(f"resuming from {RESUME_DIR} but its data cache is incomplete: {cached}")

if all(cached.values()):
    print("Loading data cache from a previous session:")
    for k, p in cached.items():
        print(f"  {k}: {p}")
        dst = os.path.join(OUTPUT_DIR, CACHE[k])
        if os.path.abspath(p) != os.path.abspath(dst):
            shutil.copy(p, dst)  # carry the cache forward so the next session's input has it too
    train_blocks = np.load(os.path.join(OUTPUT_DIR, CACHE["train"]), mmap_mode="r")
    val_blocks = np.load(os.path.join(OUTPUT_DIR, CACHE["val"]), mmap_mode="r")
    with open(os.path.join(OUTPUT_DIR, CACHE["eval"])) as f:
        eval_set = json.load(f)
    with open(os.path.join(OUTPUT_DIR, CACHE["meta"])) as f:
        data_meta = json.load(f)
    expect = {"block": BLOCK, "seed": SEED, "a2_train_tokens": A2_TRAIN_TOKENS, "a2_val_tokens": A2_VAL_TOKENS}
    bad = {k: (data_meta.get(k), v) for k, v in expect.items() if data_meta.get(k) != v}
    assert not bad, f"data cache built with different settings (cache, notebook): {bad}"
    assert train_blocks.dtype == np.uint16 and train_blocks.ndim == 2 and train_blocks.shape[1] == BLOCK and len(train_blocks) > 0
    assert val_blocks.dtype == np.uint16 and val_blocks.shape[1] == BLOCK and len(val_blocks) > 0
else:
    print("Building data: replaying Arch 2's pipeline, then collecting text it never read (session 1).")
    a2_rng = random.Random(SEED)   # reproduces Arch 2's global `random` stream (random.seed(SEED) before its loop)
    new_tok = array.array('H')
    val_tok = array.array('H')
    eval_set = []
    stats = {}
    earlier_keys = set()           # documents Arch 2 read in this or an earlier source: never reused as new data
    a2_train_ps = A2_TRAIN_TOKENS // len(TRAIN_FILES)
    a2_val_ps = A2_VAL_TOKENS // len(TRAIN_FILES)
    new_ps = NEW_TOKENS // len(TRAIN_FILES)
    eval_ps = EVAL_N // len(TRAIN_FILES)
    min_words = TARGET_WORDS + 512

    for si, train_file in enumerate(TRAIN_FILES):
        src = train_file.split("/")[-1].replace("train_", "").replace(".json", "")
        print(f"\n--- {src} ---")
        s = stats[src] = Counter()
        file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=train_file,
                                    repo_type="dataset", token=HF_TOKEN)
        b_rng = random.Random(SEED * 1000 + si)   # separate stream for the new region, so a2_rng stays exact
        seen = set()                              # per-source dedup, exactly as Arch 2
        a2_train_len = 0
        val_limit = len(val_tok) + a2_val_ps
        new_start = len(new_tok)
        new_limit = new_start + new_ps
        eval_start = len(eval_set)
        phase = "A"

        with open(file_path, 'rb') as f:
            for record in ijson.items(f, 'item'):
                s["docs_scanned"] += 1
                text = record["text"]
                h = hashlib.md5(text[:500].encode()).hexdigest()
                if h in seen:
                    continue
                seen.add(h)
                if len(text.split()) < 50:
                    continue

                if phase == "A":
                    # --- exact replay of Arch 2: same draws, same quotas, same stop point ---
                    s["A_docs"] += 1
                    toks = tokenizer.encode(text)
                    earlier_keys.add(h)
                    if a2_rng.random() < 0.05:
                        if len(val_tok) < val_limit:
                            val_tok.extend(toks[:val_limit - len(val_tok)])
                    else:
                        if a2_train_len < a2_train_ps:
                            a2_train_len += min(len(toks), a2_train_ps - a2_train_len)
                    if a2_train_len >= a2_train_ps and len(val_tok) >= val_limit:
                        phase = "B"
                        s["A_stop_doc"] = s["docs_scanned"]
                        tokenizer._encode_cache = {}
                        gc.collect()
                        print(f"  Arch 2 stopped here: doc {s['docs_scanned']:,} ({s['A_docs']:,} kept). Collecting new text...")
                    continue

                # --- region Arch 2 never read ---
                if h in earlier_keys:
                    s["B_seen_in_earlier_source"] += 1
                    continue
                s["B_docs"] += 1
                if b_rng.random() < HELDOUT_PCT / 100:
                    s["B_heldout_docs"] += 1
                    words = text.split()
                    if len(eval_set) - eval_start < eval_ps and len(words) >= min_words:
                        st = b_rng.randint(0, len(words) - min_words)
                        eval_set.append({"source": src, "doc_key": h,
                                         "prompt": " ".join(words[st:st + 512]),
                                         "reference": " ".join(words[st + 512:st + min_words])})
                    continue
                if len(new_tok) < new_limit:
                    toks = tokenizer.encode(text)
                    new_tok.extend(toks[:new_limit - len(new_tok)])
                eval_full = len(eval_set) - eval_start >= eval_ps
                if len(new_tok) >= new_limit and (eval_full or s["B_docs"] > 200_000):
                    break
                if s["docs_scanned"] % 20000 == 0:
                    print(f"  ... {s['docs_scanned']:,} docs | new={len(new_tok)-new_start:,} | eval={len(eval_set)-eval_start}")
            else:
                s["source_exhausted"] = 1

        s["A_train_tokens"] = a2_train_len
        s["new_tokens"] = len(new_tok) - new_start
        s["eval_examples"] = len(eval_set) - eval_start
        tokenizer._encode_cache = {}
        gc.collect()
        print(f"  {dict(s)}")

    del seen, earlier_keys
    n_tr = len(new_tok) // BLOCK
    n_va = len(val_tok) // BLOCK
    train_blocks = np.frombuffer(new_tok, dtype=np.uint16)[:n_tr * BLOCK].reshape(n_tr, BLOCK).copy()
    val_blocks = np.frombuffer(val_tok, dtype=np.uint16)[:n_va * BLOCK].reshape(n_va, BLOCK).copy()
    del new_tok, val_tok
    gc.collect()
    data_meta = {
        "seed": SEED, "block": BLOCK, "a2_train_tokens": A2_TRAIN_TOKENS, "a2_val_tokens": A2_VAL_TOKENS,
        "replay": "Arch 2 pipeline replayed with random.Random(SEED); per-source stop points recorded as A_stop_doc",
        "new_region": f"documents after each source's Arch 2 stop point, excluding any document Arch 2 read in this or an "
                      f"earlier source; {HELDOUT_PCT}% held out (never trained) for the eval prompts",
        "stats": {k: dict(v) for k, v in stats.items()},
        "eval_doc_keys": [e["doc_key"] for e in eval_set],
    }
    np.save(os.path.join(OUTPUT_DIR, CACHE["train"]), train_blocks)
    np.save(os.path.join(OUTPUT_DIR, CACHE["val"]), val_blocks)
    with open(os.path.join(OUTPUT_DIR, CACHE["eval"]), "w") as f:
        json.dump(eval_set, f)
    with open(os.path.join(OUTPUT_DIR, CACHE["meta"]), "w") as f:
        json.dump(data_meta, f)

print(f"\n{'='*60}\nDATA AUDIT")
for src, s in data_meta["stats"].items():
    print(f"{src:14s} Arch2 stop doc={s.get('A_stop_doc', 'exhausted')} | Arch2 train tokens={s.get('A_train_tokens', 0):,} | "
          f"new docs={s.get('B_docs', 0):,} (held out {s.get('B_heldout_docs', 0):,}, dup-of-earlier {s.get('B_seen_in_earlier_source', 0):,}) | "
          f"new tokens={s.get('new_tokens', 0):,} | eval={s.get('eval_examples', 0)} | exhausted={bool(s.get('source_exhausted'))}")
print(f"\nNew training blocks: {len(train_blocks):,} ({len(train_blocks)*BLOCK/1e6:.0f}M tokens) | "
      f"Arch 2 validation blocks: {len(val_blocks):,} | held-out eval prompts: {len(eval_set)}")


## 7. Training Blocks and Resumable Sampler

In [ ]:
tokenizer._encode_cache = {}
gc.collect()


class EpochSampler:
    """Walks a seeded permutation of block indices, reshuffling each epoch. State = (epoch, pos), so it resumes exactly."""
    def __init__(self, n, seed):
        self.n, self.seed, self.epoch, self.pos = n, seed, 0, 0
        self._perm()

    def _perm(self):
        self.perm = np.random.default_rng([self.seed, self.epoch]).permutation(self.n)

    def take(self, k):
        out = []
        while len(out) < k:
            if self.pos >= self.n:
                self.epoch += 1
                self.pos = 0
                self._perm()
            m = min(k - len(out), self.n - self.pos)
            out.extend(self.perm[self.pos:self.pos + m].tolist())
            self.pos += m
        return np.array(out, dtype=np.int64)

    def state(self):
        return {"epoch": self.epoch, "pos": self.pos}

    def load(self, st):
        self.epoch, self.pos = st["epoch"], st["pos"]
        self._perm()


train_sampler = EpochSampler(len(train_blocks), SEED + 1)


def to_xy(blocks_np):
    """(B, 1025) uint16 block -> input (B, 1024) and target (B, 1024), the same shift as Arch 2's TokenBlockDataset."""
    t = torch.from_numpy(np.ascontiguousarray(blocks_np).astype(np.int64))
    return t[:, :-1].to(device, non_blocking=True), t[:, 1:].to(device, non_blocking=True)


def make_microbatch(sampler=None):
    sampler = train_sampler if sampler is None else sampler
    return to_xy(train_blocks[sampler.take(BATCH_SIZE)])


print(f"Training blocks: {len(train_blocks):,} | one pass = {len(train_blocks) // (BATCH_SIZE * GRAD_ACCUM):,} updates "
      f"of {BATCH_SIZE * GRAD_ACCUM * MODEL_SEQ_LEN:,} tokens")


## 8. Model Architecture: GQA Attention with RoPE (unchanged from Arch 2)

In [ ]:
%%writefile attention.py
"""Grouped-Query Attention with Rotary Position Embeddings (RoPE)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def build_rope_cache(max_seq_len, d_head, base=10000.0):
    half = d_head // 2
    freqs = 1.0 / (base ** (torch.arange(0, half).float() / half))
    t = torch.arange(max_seq_len).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    T = x.shape[2]
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class CausalSelfAttention(nn.Module):

    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        cos, sin = build_rope_cache(max_seq_len, self.d_head)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out

## 9. Model Architecture: LLaMA-style Decoder (unchanged from Arch 2)

In [ ]:
%%writefile model_arch.py
"""LLaMA-style Decoder LM: RMSNorm, SwiGLU FFN, RoPE (in attention), GQA."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 12
    n_heads: int = 10
    n_kv_heads: int = 2
    d_ff: int = 1706
    max_seq_len: int = 1024
    dropout: float = 0.0


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads,
                                        cfg.max_seq_len, cfg.dropout)
        self.ln2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

## 10. Build the Model and Load Arch 2

In [ ]:
import torch.nn as nn
import sys
for _m in ("model_arch", "attention"):
    sys.modules.pop(_m, None)
from model_arch import LMConfig, DecoderLM, RMSNorm

cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=MODEL_HEADS,
    n_kv_heads=MODEL_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MODEL_SEQ_LEN,
    dropout=MODEL_DROPOUT,
)

model = DecoderLM(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
assert n_params == ARCH2_PARAMS, f"architecture mismatch: {n_params:,} params, Arch 2 has {ARCH2_PARAMS:,}"

if ARCH2_DIR:   # fresh continuation run: start from Arch 2's final weights (the model that scored 0.8334)
    state = torch.load(os.path.join(ARCH2_DIR, "model.pt"), map_location=device, weights_only=True)
    model.load_state_dict(state, strict=True)
    del state
    print(f"Loaded Arch 2 weights from {ARCH2_DIR}/model.pt")
assert model.head.weight is model.tok_emb.weight
print(f"Model parameters: {n_params:,} ({n_params/1e6:.1f}M)")

use_amp = torch.cuda.is_available()
def amp_ctx():
    return torch.autocast(device_type="cuda" if use_amp else "cpu", dtype=torch.float16, enabled=use_amp)

def GradScaler(device_type="cuda", enabled=True):
    """torch.amp.GradScaler on PyTorch >= 2.3, torch.cuda.amp.GradScaler (as in Arch 2) otherwise."""
    try:
        from torch.amp import GradScaler as _GS
        return _GS(device_type, enabled=enabled)
    except (ImportError, TypeError):
        from torch.cuda.amp import GradScaler as _GS
        return _GS(enabled=enabled)
print(f"torch {torch.__version__} | fp16 autocast: {use_amp}")


## 11. Optimizer (same groups as Arch 2) and Arch 2 AdamW State

In [ ]:
# weight decay only on 2D params (Linear weights), NOT on biases + LayerNorm
decay_params = []
no_decay_params = []

for name, param in model.named_parameters():
    if param.dim() >= 2:
        decay_params.append(param)
    else:
        no_decay_params.append(param)

print(f"Decay params:    {sum(p.numel() for p in decay_params):,}")
print(f"No-decay params: {sum(p.numel() for p in no_decay_params):,}")

optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=PEAK_LR, betas=(0.9, 0.95), eps=1e-8)
# Continue Arch 2's AdamW moments and GradScaler state (checkpoint.pt, step 9000). Param groups are built in the
# same order as Arch 2 (decay, no-decay), so the saved state maps onto the same parameters.
A2_SCALER_STATE = None
if ARCH2_DIR and os.path.exists(os.path.join(ARCH2_DIR, "checkpoint.pt")):
    _ck = torch.load(os.path.join(ARCH2_DIR, "checkpoint.pt"), map_location=device, weights_only=False)
    optimizer.load_state_dict(_ck["optimizer"])
    A2_SCALER_STATE = _ck.get("scaler")
    print(f"Loaded Arch 2 AdamW state (step {_ck.get('step')}) and GradScaler state")
    del _ck


## 12. Learning Rate: Re-warm then Cosine

In [ ]:
assert MEASURE_TO <= WARMUP_STEPS, "the horizon must be measured inside warmup, before cosine decay starts"

def lr_at(update, planned_total):
    """Re-warm linearly from START_LR (where Arch 2 ended) to PEAK_LR over WARMUP_STEPS updates, then cosine to MIN_LR
    across the planned run. planned_total is measured once during warmup and then frozen."""
    if update < WARMUP_STEPS:
        return START_LR + (PEAK_LR - START_LR) * update / WARMUP_STEPS
    assert planned_total is not None, "update horizon must be fixed before warmup ends"
    progress = min(1.0, (update - WARMUP_STEPS) / max(1, planned_total - WARMUP_STEPS))
    return MIN_LR + 0.5 * (1.0 + math.cos(math.pi * progress)) * (PEAK_LR - MIN_LR)


def plan_horizon(tp):
    per_update = float(np.mean(tp))
    total = int(TRAIN_BUDGET_HOURS * 3600 * HORIZON_SAFETY / per_update)
    return total, {"s_per_update": per_update}

print(f"LR: {START_LR:.2e} -> {PEAK_LR:.0e} over {WARMUP_STEPS} updates, cosine to {MIN_LR:.0e} over a horizon measured at "
      f"updates {MEASURE_FROM}-{MEASURE_TO} and fitted to {TRAIN_BUDGET_HOURS}h of active training.")


## 13. Generation (Arch 2, unchanged), Validation Loss and Held-out Evaluation

In [ ]:
import torch.nn.functional as F
from torch.cuda.amp import autocast   # Arch 2's generate() uses this

# ---- Arch 2's generate(), unchanged ----
def generate(model, tokenizer, prompt, max_tokens=384, target_words=128,
             temperature=0.4, top_p=0.92, top_k=50, repetition_penalty=1.15):
    model.eval()
    tokens = tokenizer.encode(prompt)
    generated = []

    with torch.no_grad():
        for _ in range(max_tokens):
            context = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
            # truncate to max seq len (sliding window)
            context = context[:, -cfg.max_seq_len:]

            with autocast():
                logits = model(context)
            logits = logits[0, -1].float()  # (vocab_size,)

            # repetition penalty: penalize tokens already generated
            for token_id in set(generated[-100:]):  # look back 100 tokens
                logits[token_id] /= repetition_penalty

            # temperature scaling
            logits = logits / temperature

            # top-k filtering
            if top_k > 0:
                top_k_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_k_vals[-1]] = float('-inf')

            # top-p (nucleus) filtering
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
            sorted_indices_to_remove[0] = False
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[indices_to_remove] = float('-inf')

            # sample from the filtered distribution
            probs = torch.softmax(logits, dim=-1)
            token_id = torch.multinomial(probs, num_samples=1).item()

            tokens.append(token_id)
            generated.append(token_id)

            # check word count
            decoded_so_far = tokenizer.decode(generated)
            if len(decoded_so_far.split()) >= target_words:
                break

    # decode all generated tokens at once and trim to target_words
    continuation = tokenizer.decode(generated)
    words = continuation.split()[:target_words]
    return " ".join(words)

def val_loss(max_batches=None):
    """Arch 2's validation arithmetic: sequential batches of 4 blocks (last partial batch kept), mean of per-batch
    mean cross-entropy. On the replayed validation set this reproduces Arch 2's reported final val loss."""
    was_training = model.training
    model.eval()
    n = len(val_blocks)
    starts = list(range(0, n, BATCH_SIZE))
    if max_batches is not None:
        starts = starts[:max_batches]
    tot = 0.0
    with torch.no_grad():
        for a in starts:
            x, y = to_xy(val_blocks[a:a + BATCH_SIZE])
            with amp_ctx():
                lg = model(x)
            tot += F.cross_entropy(lg.float().view(-1, lg.size(-1)), y.reshape(-1)).item()
    if was_training:
        model.train()
    return tot / len(starts)


def repetition_rate(text, n=4):
    w = text.split()
    grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 0.0 if not grams else 1.0 - len(set(grams)) / len(grams)


def evaluate_heldout(tag):
    """BERTScore on held-out 512 -> 128 prompts from documents neither Arch 2 nor this run trains on.
    Uses Arch 2's generate() and settings. bert_score package, roberta-large defaults: an approximation of the
    official scorer, but identical for the before/after comparison."""
    from bert_score import BERTScorer
    t0 = time.time()
    preds, errors = [], []
    for i, ex in enumerate(eval_set):
        torch.manual_seed(SEED + i)           # same sampling noise for every model evaluated
        try:
            preds.append(generate(model, tokenizer, ex["prompt"], max_tokens=MAX_GEN_TOKENS, target_words=TARGET_WORDS,
                                  temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, repetition_penalty=REPETITION_PENALTY))
        except Exception as e:
            preds.append("")
            errors.append({"idx": i, "error": repr(e)})
    scorer = BERTScorer(model_type="roberta-large", lang="en", device=str(device))
    P, R, F1 = scorer.score(preds, [ex["reference"] for ex in eval_set])
    del scorer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    vl = val_loss(MAX_VAL_BATCHES)
    res = {"tag": tag, "update": run["update"], "bertscore_P": P.mean().item(), "bertscore_R": R.mean().item(),
           "bertscore_F1": F1.mean().item(), "val_loss_capped": vl,
           "mean_words": float(np.mean([len(p.split()) for p in preds])),
           "short_outputs": sum(len(p.split()) < TARGET_WORDS for p in preds),
           "repetition_4gram": float(np.mean([repetition_rate(p) for p in preds])),
           "generation_errors": len(errors), "n": len(eval_set), "seconds": time.time() - t0,
           "per_source_F1": {s: float(np.mean([F1[i].item() for i, ex in enumerate(eval_set) if ex["source"] == s]))
                             for s in sorted({ex["source"] for ex in eval_set})}}
    with open(os.path.join(OUTPUT_DIR, f"eval_{tag}.json"), "w") as f:
        json.dump({"result": res, "errors": errors, "predictions": preds}, f)
    print(f"  >> EVAL [{tag}] update {res['update']}: BERTScore F1={res['bertscore_F1']:.4f} "
          f"(P={res['bertscore_P']:.4f} R={res['bertscore_R']:.4f}) | val loss (capped)={vl:.4f} | "
          f"short={res['short_outputs']} rep4={res['repetition_4gram']:.3f} errors={len(errors)} | {res['seconds']/60:.1f} min")
    model.train()
    return res


## 14. Pre-flight Checks (including the replay check against Arch 2)

In [ ]:
import copy

def check(name, cond, detail=""):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))
    assert cond, name

print(f"Running pre-flight checks on {device} (fp16 autocast: {use_amp})\n")

# 1. architecture and weights
check("architecture identical to Arch 2 (parameter count, tied embedding)",
      n_params == ARCH2_PARAMS and model.head.weight is model.tok_emb.weight, f"{n_params:,} params")

# 2. input/target shift identical to Arch 2's TokenBlockDataset
x, y = to_xy(train_blocks[:2])
check("target = input shifted by one token", x.shape == (2, MODEL_SEQ_LEN) and torch.equal(x[:, 1:], y[:, :-1]))

# 3. replay verification: the starting model on the replayed validation set must reproduce Arch 2's final val loss
START_VAL_LOSS = None
if ARCH2_DIR:
    START_VAL_LOSS = val_loss()
    diff = abs(START_VAL_LOSS - A2_FINAL_VAL_LOSS)
    if diff < 0.01:
        print(f"[PASS] data replay is exact: Arch 2 val loss reproduced ({START_VAL_LOSS:.4f} vs reported {A2_FINAL_VAL_LOSS:.4f})")
    else:
        print(f"[WARN] replayed val loss {START_VAL_LOSS:.4f} differs from Arch 2's {A2_FINAL_VAL_LOSS:.4f}: the corpus or "
              f"pipeline may differ from Arch 2's run. Training data is still taken after the recorded stop points.")
else:
    print("[SKIP] replay verification (resumed run; it was checked in the first session)")

# 4. new data comes only from after Arch 2's stop point in every source
stops = {s: v.get("A_stop_doc") for s, v in data_meta["stats"].items()}
check("every source has a recorded Arch 2 stop point (new text starts after it)", all(v is not None for v in stops.values()), str(stops))

# 5. sampler resumes exactly
s1 = EpochSampler(1000, 5); s1.take(1500); st = s1.state(); nxt = s1.take(50)
s2 = EpochSampler(1000, 5); s2.load(st)
check("sampler state resumes to the identical next blocks (across an epoch boundary)", np.array_equal(nxt, s2.take(50)))

# 6. save/resume on a tiny model reproduces the uninterrupted trajectory
def tiny():
    torch.manual_seed(3)
    return DecoderLM(LMConfig(vocab_size=VOCAB_SIZE, d_model=64, n_layers=2, n_heads=4, n_kv_heads=2, d_ff=176,
                              max_seq_len=MODEL_SEQ_LEN)).to(device)
tb = [to_xy(train_blocks[i * 2:(i + 1) * 2, :65]) for i in range(4)]
def tstep(m, o, b):
    o.zero_grad()
    F.cross_entropy(m(b[0]).view(-1, VOCAB_SIZE), b[1].reshape(-1)).backward()
    o.step()
mf = tiny(); of = torch.optim.AdamW(mf.parameters(), lr=1e-3)
for b in tb:
    tstep(mf, of, b)
ma = tiny(); oa = torch.optim.AdamW(ma.parameters(), lr=1e-3)
for b in tb[:2]:
    tstep(ma, oa, b)
_tmp = os.path.join(OUTPUT_DIR, "_preflight_resume.pt")
atomic_save({"m": ma.state_dict(), "o": oa.state_dict()}, _tmp)
mb = tiny(); ob = torch.optim.AdamW(mb.parameters(), lr=1e-3)
_s = torch.load(_tmp, map_location=device, weights_only=False); mb.load_state_dict(_s["m"]); ob.load_state_dict(_s["o"])
os.remove(_tmp)
for b in tb[2:]:
    tstep(mb, ob, b)
rdiff = max(float((p - q).abs().max()) for p, q in zip(mf.parameters(), mb.parameters()))
check("save/resume reproduces the uninterrupted trajectory", rdiff < 1e-5, f"max param diff {rdiff:.2e}")
del mf, ma, mb, of, oa, ob

# 7. real-size fp16 update on a copy: finite loss and gradient, parameters move
mc = copy.deepcopy(model).train()
oc = torch.optim.AdamW(mc.parameters(), lr=PEAK_LR)
sc = GradScaler("cuda", enabled=use_amp)
before = mc.blocks[0].attn.q_proj.weight.detach().clone()
skips, last_loss, gnorm = 0, None, float("nan")
for attempt in range(6):
    oc.zero_grad(set_to_none=True)
    bx, by = make_microbatch(EpochSampler(len(train_blocks), 77 + attempt))
    with amp_ctx():
        lg = mc(bx)
    loss = F.cross_entropy(lg.float().view(-1, VOCAB_SIZE), by.reshape(-1))
    sc.scale(loss).backward()
    sc.unscale_(oc)
    gnorm = float(torch.nn.utils.clip_grad_norm_(mc.parameters(), GRAD_CLIP))
    s0 = sc.get_scale(); sc.step(oc); sc.update()
    if use_amp and sc.get_scale() < s0:
        skips += 1
        continue
    last_loss = float(loss)
    break
moved = float((mc.blocks[0].attn.q_proj.weight - before).abs().max())
check("real-size update: finite loss and gradient, parameters move",
      last_loss is not None and math.isfinite(last_loss) and math.isfinite(gnorm) and moved > 0,
      f"loss {last_loss}, grad norm {gnorm:.3f}, skipped fp16 steps {skips}, max param change {moved:.2e}")
del mc, oc, sc, lg, loss
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after checks: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

# 8. BERTScore availability (fail fast here, never in the middle of training)
try:
    try:
        import bert_score  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', 'bert_score', '-q'], check=True)
    from bert_score import BERTScorer
    _sc = BERTScorer(model_type="roberta-large", lang="en", device=str(device))
    _p, _r, _f = _sc.score(["the court held that the appeal is dismissed"], ["the appeal was dismissed by the court"])
    del _sc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[PASS] BERTScore (roberta-large) loads and scores (sample F1 {_f.item():.3f})")
except Exception as e:
    EVAL_ENABLED = False
    print(f"[WARN] BERTScore unavailable ({e!r}); held-out evals disabled, training unaffected")


## 15. Continued Training (resumable)

In [ ]:
from dataclasses import asdict

def new_run_state():
    return {"update": 0, "attempts": 0, "skipped_updates": 0, "planned_total": None, "horizon_info": None, "tp": [],
            "active_train_sec": 0.0, "eval_sec": 0.0, "tokens": 0,
            "history": [], "val_history": [], "evals": [], "eval_failures": [], "sessions": [], "checkpoints": [],
            "start_val_loss": None, "baseline_done": False, "done": False, "arch2_dir": ARCH2_DIR}

def save_checkpoint(name, kind):
    run["checkpoints"].append({"name": name, "kind": kind, "update": run["update"], "unix": time.time()})
    state = {
        "kind": kind, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
        "model_config": asdict(cfg), "run": run, "sampler": train_sampler.state(),
        "rng": {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
        "lr_schedule": {"start": START_LR, "peak": PEAK_LR, "min": MIN_LR, "warmup": WARMUP_STEPS, "type": "rewarm+cosine"},
        "data_cache_files": CACHE, "tokenizer_file": "legal_bpe_16384.json",
    }
    path = os.path.join(OUTPUT_DIR, name)
    tmp = path + ".tmp"
    torch.save(state, tmp)
    if name == LATEST_CKPT and os.path.exists(path):
        os.replace(path, os.path.join(OUTPUT_DIR, PREV_CKPT))
    os.replace(tmp, path)
    atomic_json({k: v for k, v in run.items() if k != "tp"}, os.path.join(OUTPUT_DIR, "run_log.json"))
    print(f"  >> saved {name} at update {run['update']:,} ({kind})")

def do_eval(tag):
    if not EVAL_ENABLED:
        print(f"  >> eval [{tag}] skipped (BERTScore unavailable this session)")
        return
    t0 = time.time()
    try:
        run["evals"].append(evaluate_heldout(tag))
    except Exception as e:
        run["eval_failures"].append({"tag": tag, "update": run["update"], "error": repr(e)})
        print(f"  !! eval [{tag}] failed ({e!r}); training continues")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    finally:
        model.train()
        run["eval_sec"] += time.time() - t0

# ---------------- state / resume ----------------
scaler = GradScaler("cuda", enabled=use_amp)
run = new_run_state()
shutil.copy(TOKENIZER_PATH, os.path.join(OUTPUT_DIR, "legal_bpe_16384.json"))

if RESUME_DIR and os.path.abspath(RESUME_DIR) != os.path.abspath(OUTPUT_DIR):
    for name in [PREV_CKPT] + [os.path.basename(p) for p in glob.glob(os.path.join(RESUME_DIR, "eval_*.json"))
                               + glob.glob(os.path.join(RESUME_DIR, "arch2c_snap_*.pt"))]:
        src = os.path.join(RESUME_DIR, name)
        if os.path.exists(src) and not os.path.exists(os.path.join(OUTPUT_DIR, name)):
            shutil.copy(src, os.path.join(OUTPUT_DIR, name))

ck, resume_path = None, None
if RESUME_DIR:
    for name in (LATEST_CKPT, PREV_CKPT):
        p = os.path.join(RESUME_DIR, name)
        if not os.path.exists(p):
            continue
        try:
            ck, resume_path = torch.load(p, map_location=device, weights_only=False), p
            break
        except Exception as e:
            print(f"  !! could not load {p} ({e!r}); trying the fallback checkpoint")
    if ck is None:
        raise RuntimeError(f"no loadable continuation checkpoint in {RESUME_DIR}")

if ck is not None:
    print(f"Resuming from {resume_path}")
    assert ck["model_config"] == asdict(cfg), "architecture config differs from the checkpoint"
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    run = ck["run"]
    for k, v in new_run_state().items():
        run.setdefault(k, v)
    train_sampler.load(ck["sampler"])
    random.setstate(ck["rng"]["python"]); np.random.set_state(ck["rng"]["numpy"])
    torch.set_rng_state(ck["rng"]["torch"].cpu())            # RNG states must be CPU ByteTensors; map_location moved them
    if torch.cuda.is_available() and ck["rng"]["cuda"] is not None:
        torch.cuda.set_rng_state_all([s.cpu() for s in ck["rng"]["cuda"]])
    del ck
    print(f"  update {run['update']:,} / planned {run['planned_total']} | active training so far "
          f"{run['active_train_sec']/3600:.2f}h | skipped fp16 updates {run['skipped_updates']}")
else:
    if A2_SCALER_STATE is not None and use_amp:
        scaler.load_state_dict(A2_SCALER_STATE)
    run["start_val_loss"] = START_VAL_LOSS
    print(f"Starting continued pretraining from Arch 2 (val loss at start: {START_VAL_LOSS})")

run["sessions"].append({"start_unix": time.time(), "start_update": run["update"], "data_ready_after_sec": session_elapsed()})
session_deadline = (SESSION_LIMIT_HOURS - SESSION_RESERVE_HOURS) * 3600
last_ckpt = time.time()
win_loss, win_n = 0.0, 0
consecutive_skips = 0
stop_reason = None
stage = "idle"
model.train()

if not run["baseline_done"] and run["update"] == 0:
    do_eval("arch2_original")       # the original Arch 2 on the held-out prompts, before any further training
    run["baseline_done"] = True

try:
    while True:
        stage = "idle"
        u = run["update"]
        if run["planned_total"] is not None and u >= run["planned_total"]:
            run["done"] = True
            stop_reason = "planned update horizon reached"
            break
        if session_elapsed() >= session_deadline:
            stop_reason = f"session safety cutoff ({SESSION_LIMIT_HOURS - SESSION_RESERVE_HOURS:.2f}h of session time)"
            break

        t0 = time.time()
        stage = "batch"
        lr = lr_at(u, run["planned_total"])
        for pg in optimizer.param_groups:
            pg["lr"] = lr
        micro = [train_blocks[train_sampler.take(BATCH_SIZE)] for _ in range(GRAD_ACCUM)]

        stage = "forward"
        optimizer.zero_grad(set_to_none=True)
        loss_sum = 0.0
        for blk in micro:
            bx, by = to_xy(blk)
            with amp_ctx():
                logits = model(bx)
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), by.reshape(-1)) / GRAD_ACCUM
            scaler.scale(loss).backward()        # every position is a real token, so the mean is over all of them
            loss_sum += loss.item()
            del logits, loss
        scaler.unscale_(optimizer)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP))
        scale_before = scaler.get_scale()
        stage = "step"
        scaler.step(optimizer)
        scaler.update()
        if use_amp:
            torch.cuda.synchronize()
        dt = time.time() - t0
        run["active_train_sec"] += dt
        run["attempts"] += 1

        if (use_amp and scaler.get_scale() < scale_before) or not math.isfinite(loss_sum):
            run["skipped_updates"] += 1
            consecutive_skips += 1
            stage = "idle"
            if consecutive_skips >= 50:
                raise RuntimeError(f"{consecutive_skips} consecutive skipped updates at update {u}; training is unstable")
            continue
        consecutive_skips = 0
        run["update"] = u + 1
        stage = "idle"
        run["tokens"] += BATCH_SIZE * GRAD_ACCUM * MODEL_SEQ_LEN
        win_loss += loss_sum; win_n += 1

        if MEASURE_FROM <= u < MEASURE_TO:
            run["tp"].append(dt)
        if run["planned_total"] is None and run["update"] >= MEASURE_TO:
            total, info = plan_horizon(run["tp"])
            assert total > 2 * WARMUP_STEPS, f"horizon {total} too short; check throughput"
            run.update(planned_total=total, horizon_info=info)
            per = info["s_per_update"]
            print(f"\n  >> HORIZON FIXED: {total:,} updates = {total * BATCH_SIZE * GRAD_ACCUM * MODEL_SEQ_LEN / 1e6:,.0f}M new tokens "
                  f"at {per:.2f}s/update -> {total*per/3600:.1f}h active training "
                  f"(15% slower/faster: {total*per*1.15/3600:.1f}h / {total*per*0.85/3600:.1f}h).\n")

        if run["update"] % LOG_EVERY == 0:
            print(f"upd {run['update']:>6,}/{run['planned_total'] or '?'} | loss {win_loss/win_n:.4f} | lr {lr:.2e} | "
                  f"gnorm {gnorm:.2f} | scale {scaler.get_scale():.0f} | skipped {run['skipped_updates']} | {dt:.2f}s/upd | "
                  f"{run['tokens']/1e6:,.0f}M new tokens | train {run['active_train_sec']/3600:.2f}h session {session_elapsed()/3600:.2f}h")
            run["history"].append({"update": run["update"], "loss": win_loss / win_n, "lr": lr, "gnorm": gnorm,
                                   "tokens": run["tokens"], "active_train_sec": run["active_train_sec"]})
            win_loss, win_n = 0.0, 0

        if run["update"] % VAL_EVERY_UPDATES == 0:
            tv = time.time()
            vl = val_loss(MAX_VAL_BATCHES)
            run["eval_sec"] += time.time() - tv
            run["val_history"].append({"update": run["update"], "val_loss_capped": vl})
            print(f"  >> val loss (first {MAX_VAL_BATCHES} batches) {vl:.4f} (ppl {math.exp(min(vl, 20)):.2f})")

        if run["update"] % CKPT_EVERY_UPDATES == 0 or time.time() - last_ckpt >= CKPT_EVERY_MIN * 60:
            save_checkpoint(LATEST_CKPT, "latest")
            last_ckpt = time.time()

        if (run["planned_total"] is not None and run["update"] % SNAPSHOT_EVERY == 0
                and run["update"] >= SNAPSHOT_FROM_FRAC * run["planned_total"]):
            snap = os.path.join(OUTPUT_DIR, f"arch2c_snap_{run['update']:06d}.pt")
            atomic_save({k: v.half() for k, v in model.state_dict().items()}, snap)
            print(f"  >> weight snapshot {os.path.basename(snap)}")

except BaseException as e:
    print(f"\n!! training interrupted by {type(e).__name__}: {e}")
    if stage != "step":
        try:
            save_checkpoint(LATEST_CKPT, f"emergency save after {type(e).__name__}")
        except BaseException as e2:
            print(f"!! emergency save failed: {e2!r}")
    else:
        print("!! interrupted inside the optimizer step; keeping the last regular checkpoint")
    raise

print(f"\nStopped: {stop_reason}")
run["sessions"][-1].update({"end_unix": time.time(), "end_update": run["update"], "session_elapsed_sec": session_elapsed()})
save_checkpoint(LATEST_CKPT, "latest (end of session)")
TRAINING_COMPLETE = run["done"]

if TRAINING_COMPLETE:
    atomic_save(model.state_dict(), os.path.join(OUTPUT_DIR, "model.pt"))   # before any generation
    print("model.pt saved.")
    do_eval("continued_final")
    final_val_loss = val_loss()
    final_ppl = math.exp(min(final_val_loss, 20))
    run["final_val_loss"] = final_val_loss
    save_checkpoint(LATEST_CKPT, "latest (training complete)")
    print(f"Val loss on Arch 2's validation set: start {run['start_val_loss']} -> end {final_val_loss:.4f} (ppl {final_ppl:.2f})")
    for e in run["evals"]:
        print(f"  held-out BERTScore F1 [{e['tag']}] = {e['bertscore_F1']:.4f}")
else:
    print(f"Training not finished: update {run['update']:,} of {run['planned_total']}. To resume: in a new version, add "
          f"this version's output as an input, set RUN_MODE = \"resume\", and Save & Run All.")
print(f"Active training {run['active_train_sec']/3600:.2f}h | eval {run['eval_sec']/3600:.2f}h | "
      f"new tokens {run['tokens']/1e6:,.0f}M | skipped fp16 updates {run['skipped_updates']}")


## 16. Training Curves and Before/After Evaluation

In [ ]:
import matplotlib.pyplot as plt

h = run["history"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
if h:
    axes[0].plot([x["update"] for x in h], [x["loss"] for x in h])
axes[0].set_title("Training loss (new text)"); axes[0].set_xlabel("update"); axes[0].grid(alpha=0.3)
if run["val_history"]:
    axes[1].plot([v["update"] for v in run["val_history"]], [v["val_loss_capped"] for v in run["val_history"]], marker="o", color="tab:red")
if run.get("start_val_loss"):
    axes[1].axhline(run["start_val_loss"], color="gray", ls="--", label=f"Arch 2 (start) {run['start_val_loss']:.3f}")
    axes[1].legend()
axes[1].set_title("Validation loss (Arch 2's validation set)"); axes[1].set_xlabel("update"); axes[1].grid(alpha=0.3)
if h:
    axes[2].plot([x["update"] for x in h], [x["lr"] for x in h])
axes[2].set_title("Learning rate (re-warm + cosine)"); axes[2].set_xlabel("update"); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=150)
plt.show()
for e in run["evals"]:
    print(f"eval {e['tag']:>16s} @ {e['update']:>6,}: held-out BERTScore F1 {e['bertscore_F1']:.4f} | rep4 {e['repetition_4gram']:.3f}")


## 17. Load Test Data

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Number of test examples: {len(test_data)}")

# show a sample prompt
sample_id = list(test_data.keys())[0]
print(f"\nSample prompt (id={sample_id}):")
print(test_data[sample_id]["text"][:300] + "...")

## 18. Generate Test Predictions (only when training is complete)

In [ ]:
predictions = {}
if not TRAINING_COMPLETE:
    print("Training is not complete; skipping test generation this session.")
else:
    items = list(test_data.items())
    total_examples = len(items)
    print(f"Generating {total_examples} predictions with Arch 2's generate() and settings...")
    t0 = time.time()
    for i, (example_id, record) in enumerate(items):
        prompt = record["text"]
        prediction = generate(
            model, tokenizer, prompt,
            max_tokens=MAX_GEN_TOKENS,
            target_words=TARGET_WORDS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            repetition_penalty=REPETITION_PENALTY,
        )
        predictions[str(example_id)] = {"text": prediction}
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{total_examples} ({100*(i+1)/total_examples:.1f}%) | {time.time()-t0:.0f}s")
    print(f"\nGeneration complete in {(time.time()-t0)/60:.1f} min")


## 19. Sample Predictions

In [ ]:
for i, (eid, pred) in enumerate(list(predictions.items())[:5]):
    prompt = test_data[eid]["text"]
    print(f"\n{'='*60}")
    print(f"Example {eid}")
    print(f"PROMPT: {prompt[:200]}...")
    print(f"GENERATED ({len(pred['text'].split())} words): {pred['text'][:300]}...")


## 20. Save Predictions & Create Submission

In [ ]:
if TRAINING_COMPLETE:
    PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
    SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(PREDICTION_PATH, arcname="prediction.json")
    avg_words = sum(len(p['text'].split()) for p in predictions.values()) / len(predictions)
    print(f"submission.zip created: {SUBMISSION_ZIP} | predictions: {len(predictions)} | avg words {avg_words:.1f}")
else:
    print("Training is not complete; no submission this session.")


## 21. Save Metrics

In [ ]:
metrics = {
    "architecture": "llama_style_rope_rmsnorm_swiglu_gqa_continued",
    "base_model": "Arch 2 (llama_style_rope_rmsnorm_swiglu_gqa), 300M tokens, val loss 2.2833",
    "params": n_params, "model_config": asdict(cfg),
    "training_complete": TRAINING_COMPLETE,
    "new_tokens": run["tokens"], "total_tokens_incl_arch2": A2_TRAIN_TOKENS + run["tokens"],
    "updates": run["update"], "planned_total": run["planned_total"], "horizon_info": run["horizon_info"],
    "lr_schedule": {"start": START_LR, "peak": PEAK_LR, "min": MIN_LR, "warmup": WARMUP_STEPS},
    "skipped_updates": run["skipped_updates"], "active_train_hours": run["active_train_sec"] / 3600,
    "eval_hours": run["eval_sec"] / 3600, "sessions": run["sessions"],
    "start_val_loss": run.get("start_val_loss"), "final_val_loss": run.get("final_val_loss"),
    "val_history": run["val_history"], "evals": run["evals"], "eval_failures": run["eval_failures"],
    "generation_config": {"temperature": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K,
                          "repetition_penalty": REPETITION_PENALTY, "function": "Arch 2 generate(), unchanged"},
}
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2, default=float)
print(f"metrics.json saved | training complete: {TRAINING_COMPLETE} | update {run['update']:,}/{run['planned_total']}")
print(f"Files in {OUTPUT_DIR}: {sorted(os.listdir(OUTPUT_DIR))}")
